# Search Strategy Calibration

This notebook is the search strategy calibration layer. It runs deterministic parameter sweeps over the production search stack, varies branch width, search depth, repair attempts, and retrieval depth only, logs structured experiment results, checks repeat stability and cross-fold generalization, identifies diminishing returns, and writes inspectable calibration artifacts plus recommended config overrides.


In [ ]:
from __future__ import annotations

import csv
import json
import math
import random
import shutil
from collections import Counter
from itertools import product
from pathlib import Path

import pandas as pd

from src.branches.branch_controller import BranchController, ControllerConfig
from src.common.metrics import canonical_match, duration_seconds_from_ns, monotonic_time_ns, stable_mean
from src.common.schemas import BudgetPlan, ParsedProblem, RouteDecision
from src.operators.library import OperatorLibrary
from src.operators.priors import OperatorPriorShaper
from src.parsing.parser import ProblemParser
from src.retrieval.embedder import MathEmbedder
from src.retrieval.index_builder import RetrievalIndexBuilder
from src.retrieval.query import TraceRetriever
from src.retrieval.retrieval_policy import RetrievalPolicy
from src.routing.dual_router import DualRouter
from src.state_graph.state_init import StateGraphInitializer

SEED = 1337
SAMPLE_SIZE = 24
NUM_FOLDS = 3
REPEATS = 2
OVERWRITE_OUTPUTS = True
MAX_RUNTIME_CV = 0.35
MAX_FOLD_SOLVE_GAP = 0.25
MIN_DIMINISHING_GAIN = 0.01

GRID_WIDTHS = (4, 8)
GRID_DEPTHS = (4, 6)
GRID_REPAIRS = (0, 1)
GRID_RETRIEVALS = (1, 3)

RANDOM_WIDTHS = (3, 5, 7, 9)
RANDOM_DEPTHS = (3, 5, 7)
RANDOM_REPAIRS = (0, 1, 2)
RANDOM_RETRIEVALS = (0, 2, 4)
RANDOM_SEARCH_COUNT = 6

ROOT = Path.cwd()
DATASET_PATH = ROOT / 'data' / 'raw' / 'aimo3_train.csv'
INDEX_PATH = ROOT / 'data' / 'interim' / 'retrieval_index'
RUN_ID = f'search_calibration_seed{SEED}_n{SAMPLE_SIZE}_f{NUM_FOLDS}'
OUT_DIR = ROOT / 'artifacts' / 'search_calibration' / RUN_ID
CALIBRATION_RESULTS_PATH = OUT_DIR / 'calibration_results.parquet'
EXPERIMENT_LOG_PATH = OUT_DIR / 'experiment_logs.parquet'
SUMMARY_PATH = OUT_DIR / 'search_calibration_summary.json'
OVERRIDES_PATH = OUT_DIR / 'recommended_config_overrides.json'

REQUIRED_COLUMNS = ('id', 'problem', 'answer')
MAX_RETRIEVAL_DEPTH = max(max(GRID_RETRIEVALS), max(RANDOM_RETRIEVALS))

random.seed(SEED)


def need(condition, message):
    if not condition:
        raise AssertionError(message)


def sj(value):
    return json.dumps(value, ensure_ascii=True, sort_keys=True, default=str)


def text(value):
    return ' '.join(str(value or '').split())


def model_to_data(value):
    if value is None:
        return None
    if hasattr(value, 'model_dump'):
        return value.model_dump(mode='json')
    if hasattr(value, '__dict__'):
        return dict(value.__dict__)
    return value


def fnum(value, default=0.0):
    try:
        return float(value)
    except Exception:
        return float(default)


def prepare_output_dir(path, overwrite):
    if path.exists():
        if not overwrite:
            raise FileExistsError(f'Output directory already exists: {path}')
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def load_dataset_rows(path):
    need(path.exists(), f'Dataset not found: {path}')
    need(path.stat().st_size > 0, f'Dataset is empty: {path}')
    with path.open('r', encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        fieldnames = tuple(reader.fieldnames or ())
        missing = [name for name in REQUIRED_COLUMNS if name not in fieldnames]
        need(not missing, f'Dataset schema missing required columns: {missing}; found={fieldnames}')
        rows = list(reader)
    need(rows, 'Dataset has a header but no records.')
    for index, row in enumerate(rows):
        need(text(row.get('id')), f'Row {index} has empty id.')
        need(text(row.get('problem')), f'Row {index} has empty problem text.')
        need(text(row.get('answer')), f'Row {index} has empty answer.')
    return rows, fieldnames


def sample_rows(rows, sample_size, seed):
    if len(rows) <= sample_size:
        return sorted(list(rows), key=lambda row: text(row['id']))
    picked = list(rows)
    rng = random.Random(seed)
    rng.shuffle(picked)
    return sorted(picked[:sample_size], key=lambda row: text(row['id']))


def norm_entropy(probabilities):
    values = [float(v) for v in dict(probabilities or {}).values() if float(v) > 0.0]
    if not values:
        return 0.0
    total = sum(values)
    need(total > 0.0, 'Probability mass must be positive.')
    probs = [v / total for v in values]
    entropy = -sum(p * math.log(p, 2) for p in probs if p > 0.0)
    denom = math.log(len(probs), 2) if len(probs) > 1 else 1.0
    return float(entropy / denom if denom else 0.0)


def validate_parsed(parsed):
    need(isinstance(parsed, ParsedProblem), f'Parser must return ParsedProblem, got {type(parsed)!r}.')
    need(bool(parsed.problem_id), 'ParsedProblem.problem_id is empty.')
    need(bool(str(parsed.raw_text).strip()), f'ParsedProblem.raw_text empty for {parsed.problem_id}.')
    need(parsed.canonical_problem_graph is not None, f'Canonical problem graph missing for {parsed.problem_id}.')
    need(bool(parsed.constraints or parsed.knowns or parsed.unknowns), f'ParsedProblem is structurally empty for {parsed.problem_id}.')
    return {
        'constraint_count': len(parsed.constraints),
        'known_count': len(parsed.knowns),
        'unknown_count': len(parsed.unknowns),
        'likely_archetypes': list(parsed.likely_archetypes),
    }


def validate_route(route):
    need(isinstance(route, RouteDecision), f'Router must return RouteDecision, got {type(route)!r}.')
    need(route.branch_budget > 0, f'Branch budget must be positive for {route.problem_id}.')
    need(route.retrieval_depth >= 0, f'Retrieval depth must be non-negative for {route.problem_id}.')
    need(getattr(route.budget_plan, 'max_search_depth', 0) > 0, f'Max search depth must be positive for {route.problem_id}.')
    need(getattr(route.budget_plan, 'max_search_nodes', 0) > 0, f'Max search nodes must be positive for {route.problem_id}.')
    return {
        'difficulty': getattr(getattr(route, 'difficulty', None), 'value', str(getattr(route, 'difficulty', ''))),
        'difficulty_score': float(route.difficulty_score),
        'routing_entropy': norm_entropy(route.archetype_probs),
        'budget_plan': model_to_data(route.budget_plan),
    }


def validate_state_graph(init_result):
    graph = init_result.graph
    root = init_result.root_node
    need(graph.node_count >= 1, f'State graph is empty for {root.provenance.problem_id}.')
    need(graph.get_node(root.node_id) is not None, f'Root node missing from graph for {root.provenance.problem_id}.')
    need(bool(root.constraints or root.invariants or root.goals), f'Root node is missing constraints, invariants, and goals for {root.provenance.problem_id}.')
    return {
        'root_node_id': root.node_id,
        'node_count': graph.node_count,
        'edge_count': graph.edge_count,
    }


def fold_assignments(sampled_rows, num_folds):
    assignments = {}
    for index, row in enumerate(sorted(sampled_rows, key=lambda item: text(item['id']))):
        assignments[text(row['id'])] = index % num_folds
    return assignments


def config_id(config):
    return f"bw{config['branch_width']}_d{config['depth']}_r{config['repair_attempts']}_ret{config['retrieval_depth']}"


def build_experiment_configs(seed):
    grid = []
    for branch_width, depth, repair_attempts, retrieval_depth in product(GRID_WIDTHS, GRID_DEPTHS, GRID_REPAIRS, GRID_RETRIEVALS):
        config = {
            'search_strategy': 'grid',
            'branch_width': int(branch_width),
            'depth': int(depth),
            'repair_attempts': int(repair_attempts),
            'retrieval_depth': int(retrieval_depth),
        }
        config['config_id'] = config_id(config)
        grid.append(config)

    seen = {item['config_id'] for item in grid}
    random_pool = []
    for branch_width, depth, repair_attempts, retrieval_depth in product(RANDOM_WIDTHS, RANDOM_DEPTHS, RANDOM_REPAIRS, RANDOM_RETRIEVALS):
        config = {
            'search_strategy': 'random',
            'branch_width': int(branch_width),
            'depth': int(depth),
            'repair_attempts': int(repair_attempts),
            'retrieval_depth': int(retrieval_depth),
        }
        config['config_id'] = config_id(config)
        if config['config_id'] not in seen:
            random_pool.append(config)

    rng = random.Random(seed)
    rng.shuffle(random_pool)
    selected_random = random_pool[:RANDOM_SEARCH_COUNT]
    return grid + sorted(selected_random, key=lambda item: item['config_id'])


def tuned_route(base_route, config):
    budget_data = base_route.budget_plan.model_dump(mode='json')
    derived_max_nodes = max(int(budget_data.get('max_search_nodes', 0)), int(config['branch_width']) * max(2, int(config['depth'])) * 2)
    budget_data.update(
        {
            'branch_budget': int(config['branch_width']),
            'retrieval_depth': int(config['retrieval_depth']),
            'max_search_depth': int(config['depth']),
            'max_search_nodes': int(derived_max_nodes),
            'repair_budget': int(config['repair_attempts']),
            'self_consistency_samples': int(config['branch_width']),
        }
    )
    tuned_budget = BudgetPlan(**budget_data)
    tuned = base_route.model_copy(
        update={
            'budget_plan': tuned_budget,
            'branch_budget': tuned_budget.branch_budget,
            'retrieval_depth': tuned_budget.retrieval_depth,
            'use_retrieval': bool(base_route.use_retrieval and tuned_budget.retrieval_depth > 0),
            'use_symbolic': tuned_budget.use_symbolic,
            'use_brute_force': tuned_budget.use_brute_force,
            'diagnostics': {
                **dict(base_route.diagnostics or {}),
                'search_calibration_override': {
                    'branch_width': int(config['branch_width']),
                    'depth': int(config['depth']),
                    'repair_attempts': int(config['repair_attempts']),
                    'retrieval_depth': int(config['retrieval_depth']),
                },
            },
        }
    )
    return tuned


def build_controller(operator_library, prior_shaper, config):
    max_nodes = max(int(config['branch_width']) * max(2, int(config['depth'])) * 2, int(config['branch_width']) * 4)
    return BranchController(
        operator_library=operator_library,
        prior_shaper=prior_shaper,
        config=ControllerConfig(
            deterministic=True,
            self_consistency_samples=int(config['branch_width']),
            frontier_width=int(config['branch_width']),
            max_search_depth=int(config['depth']),
            max_search_nodes=int(max_nodes),
            retain_top_k=max(2, min(int(config['branch_width']), 8)),
            repair_budget=int(config['repair_attempts']),
            resample_budget=1,
            critique_top_k=max(1, min(2, int(config['branch_width']))),
        ),
    )


def outcome_signature(frame):
    ordered = frame.sort_values('problem_id', kind='stable')
    parts = []
    for row in ordered.to_dict(orient='records'):
        parts.append(
            ':'.join(
                [
                    text(row.get('problem_id')),
                    text(row.get('status')),
                    text(row.get('predicted_answer')),
                    str(int(bool(row.get('solved')))),
                    text(row.get('stopped_reason')),
                ]
            )
        )
    return '|'.join(parts)


def runtime_cv(values):
    numeric = [float(value) for value in values if value is not None]
    if not numeric:
        return 0.0
    mean_value = stable_mean(numeric)
    if mean_value <= 0.0:
        return 0.0
    variance = sum((value - mean_value) ** 2 for value in numeric) / len(numeric)
    return math.sqrt(variance) / mean_value


def diminishing_returns(axis_frame, parameter_name):
    ordered = axis_frame.sort_values(parameter_name, kind='stable').reset_index(drop=True)
    transitions = []
    cutoff = None
    previous_efficiency = None
    for index in range(1, len(ordered)):
        prev_row = ordered.iloc[index - 1]
        row = ordered.iloc[index]
        solve_gain = float(row['solve_rate_mean']) - float(prev_row['solve_rate_mean'])
        runtime_gain = float(row['runtime_mean_sec']) - float(prev_row['runtime_mean_sec'])
        efficiency = solve_gain / runtime_gain if runtime_gain > 0 else float('inf')
        diminishing = bool(solve_gain <= MIN_DIMINISHING_GAIN or (previous_efficiency is not None and efficiency < previous_efficiency and solve_gain <= 0.02))
        if cutoff is None and diminishing:
            cutoff = int(row[parameter_name])
        transitions.append(
            {
                'from': int(prev_row[parameter_name]),
                'to': int(row[parameter_name]),
                'solve_gain': solve_gain,
                'runtime_gain': runtime_gain,
                'efficiency': None if efficiency == float('inf') else efficiency,
                'diminishing': diminishing,
            }
        )
        if efficiency != float('inf'):
            previous_efficiency = efficiency
    return {
        'parameter': parameter_name,
        'value_summary': ordered.to_dict(orient='records'),
        'transitions': transitions,
        'diminishing_returns_from': cutoff,
    }


def is_pareto_optimal(frame, index):
    row = frame.loc[index]
    for other_index, other in frame.iterrows():
        if other_index == index:
            continue
        dominates = (
            float(other['solve_rate_mean']) >= float(row['solve_rate_mean'])
            and float(other['verifier_score_mean']) >= float(row['verifier_score_mean'])
            and float(other['runtime_mean_sec']) <= float(row['runtime_mean_sec'])
            and (
                float(other['solve_rate_mean']) > float(row['solve_rate_mean'])
                or float(other['verifier_score_mean']) > float(row['verifier_score_mean'])
                or float(other['runtime_mean_sec']) < float(row['runtime_mean_sec'])
            )
        )
        if dominates:
            return False
    return True


def tradeoff_score(frame):
    solve_min = float(frame['solve_rate_mean'].min())
    solve_max = float(frame['solve_rate_mean'].max())
    verifier_min = float(frame['verifier_score_mean'].min())
    verifier_max = float(frame['verifier_score_mean'].max())
    runtime_min = float(frame['runtime_mean_sec'].min())
    runtime_max = float(frame['runtime_mean_sec'].max())

    def normalize(value, low, high, invert=False):
        if abs(high - low) <= 1e-12:
            score = 1.0
        else:
            score = (float(value) - low) / (high - low)
        score = max(0.0, min(1.0, score))
        return 1.0 - score if invert else score

    out = []
    for row in frame.to_dict(orient='records'):
        solve_component = normalize(row['solve_rate_mean'], solve_min, solve_max)
        verifier_component = normalize(row['verifier_score_mean'], verifier_min, verifier_max)
        runtime_component = normalize(row['runtime_mean_sec'], runtime_min, runtime_max, invert=True)
        stability_component = max(0.0, 1.0 - float(row['solve_rate_fold_gap']))
        out.append(0.55 * solve_component + 0.20 * verifier_component + 0.15 * runtime_component + 0.10 * stability_component)
    return out


In [ ]:
prepare_output_dir(OUT_DIR, OVERWRITE_OUTPUTS)

rows, schema = load_dataset_rows(DATASET_PATH)
sampled_rows = sample_rows(rows, SAMPLE_SIZE, SEED)
fold_by_problem_id = fold_assignments(sampled_rows, NUM_FOLDS)
experiment_configs = build_experiment_configs(SEED)
need(experiment_configs, 'No calibration configs were constructed.')

parser = ProblemParser()
router = DualRouter()
state_initializer = StateGraphInitializer()
embedder = MathEmbedder()
index = RetrievalIndexBuilder(embedder=embedder, index_path=str(INDEX_PATH))
need(index.load(), f'Retrieval index failed to load from {INDEX_PATH}.')
retrieval_policy = RetrievalPolicy()
retriever = TraceRetriever(embedder=embedder, index=index, policy=retrieval_policy)
operator_library = OperatorLibrary()
prior_shaper = OperatorPriorShaper(operator_library)

problem_contexts = []
context_failures = []
for sample_index, row in enumerate(sampled_rows):
    problem_id = text(row['id'])
    expected_answer = text(row['answer'])
    try:
        parsed = parser.parse_sync(text(row['problem']), problem_id=problem_id)
        parse_info = validate_parsed(parsed)
        route = router.route(parsed)
        route_info = validate_route(route)
        init_result = state_initializer.initialize(parsed, route)
        state_info = validate_state_graph(init_result)
        query = retriever.build_query(parsed, route, reasoning_state=init_result.root_node)
        hits = retriever.retrieve_hits(query, route, top_k=MAX_RETRIEVAL_DEPTH)
        problem_contexts.append(
            {
                'problem_id': problem_id,
                'expected_answer': expected_answer,
                'fold_index': int(fold_by_problem_id[problem_id]),
                'sample_index': sample_index,
                'parsed': parsed,
                'route': route,
                'root_node': init_result.root_node,
                'query': query,
                'retrieval_hits': tuple(hits),
                'parse_info': parse_info,
                'route_info': route_info,
                'state_info': state_info,
            }
        )
    except Exception as exc:
        context_failures.append({'problem_id': problem_id, 'error': repr(exc)})

need(problem_contexts, 'No problem contexts were prepared for search calibration.')
need(not context_failures, f'Upstream context preparation failed for {len(context_failures)} problems: {context_failures[:5]}')
need(len({context['fold_index'] for context in problem_contexts}) == NUM_FOLDS, 'Deterministic fold partition is incomplete.')

experiment_logs = []
for config in experiment_configs:
    for repeat_index in range(REPEATS):
        controller = build_controller(operator_library, prior_shaper, config)
        for context in problem_contexts:
            tuned = tuned_route(context['route'], config)
            retrieval_hits = list(context['retrieval_hits'][: int(config['retrieval_depth'])]) if tuned.use_retrieval else []
            retrieved_traces = [retrieval_policy.hit_to_trace(hit) for hit in retrieval_hits]
            start_ns = monotonic_time_ns()
            try:
                result = controller.solve(
                    problem=context['parsed'],
                    route=tuned,
                    root_node=context['root_node'],
                    retrieved_traces=retrieved_traces,
                )
                runtime_sec = duration_seconds_from_ns(start_ns)
                final_prediction = result.final_prediction
                predicted_answer = str(final_prediction.final_answer)
                solved = canonical_match(context['expected_answer'], predicted_answer)
                verifier_score = fnum(final_prediction.winning_cluster.verifier_score)
                surviving_branch_verifier_mean = stable_mean([branch.score_breakdown.verifier_probability for branch in result.surviving_branches])
                experiment_logs.append(
                    {
                        'run_id': RUN_ID,
                        'config_id': config['config_id'],
                        'search_strategy': config['search_strategy'],
                        'branch_width': int(config['branch_width']),
                        'depth': int(config['depth']),
                        'repair_attempts': int(config['repair_attempts']),
                        'retrieval_depth': int(config['retrieval_depth']),
                        'repeat_index': int(repeat_index),
                        'fold_index': int(context['fold_index']),
                        'problem_id': context['problem_id'],
                        'expected_answer': context['expected_answer'],
                        'predicted_answer': predicted_answer,
                        'status': 'ok',
                        'solved': bool(solved),
                        'runtime_sec': float(runtime_sec),
                        'verifier_score': float(verifier_score),
                        'surviving_branch_verifier_mean': float(surviving_branch_verifier_mean),
                        'confidence': float(final_prediction.confidence),
                        'num_branches_generated': int(final_prediction.num_branches_generated),
                        'num_branches_survived': int(final_prediction.num_branches_survived),
                        'retrieval_hits_used': int(len(retrieval_hits)),
                        'route_uncertainty': float(tuned.route_uncertainty),
                        'difficulty_score': float(tuned.difficulty_score),
                        'stopped_reason': text(result.stopped_reason),
                        'winning_cluster_size': int(final_prediction.winning_cluster.cluster_size),
                        'search_history_length': int(len(result.search_history)),
                        'metadata': sj({'winning_branch_ids': list(result.metadata.get('winning_branch_ids', ())), 'route_budget': model_to_data(tuned.budget_plan)}),
                    }
                )
            except Exception as exc:
                runtime_sec = duration_seconds_from_ns(start_ns)
                experiment_logs.append(
                    {
                        'run_id': RUN_ID,
                        'config_id': config['config_id'],
                        'search_strategy': config['search_strategy'],
                        'branch_width': int(config['branch_width']),
                        'depth': int(config['depth']),
                        'repair_attempts': int(config['repair_attempts']),
                        'retrieval_depth': int(config['retrieval_depth']),
                        'repeat_index': int(repeat_index),
                        'fold_index': int(context['fold_index']),
                        'problem_id': context['problem_id'],
                        'expected_answer': context['expected_answer'],
                        'predicted_answer': None,
                        'status': 'execution_failure',
                        'solved': False,
                        'runtime_sec': float(runtime_sec),
                        'verifier_score': 0.0,
                        'surviving_branch_verifier_mean': 0.0,
                        'confidence': 0.0,
                        'num_branches_generated': 0,
                        'num_branches_survived': 0,
                        'retrieval_hits_used': int(len(retrieval_hits)),
                        'route_uncertainty': float(tuned.route_uncertainty),
                        'difficulty_score': float(tuned.difficulty_score),
                        'stopped_reason': 'execution_failure',
                        'winning_cluster_size': 0,
                        'search_history_length': 0,
                        'metadata': sj({'error': repr(exc), 'route_budget': model_to_data(tuned.budget_plan)}),
                    }
                )

experiment_log_frame = pd.DataFrame(experiment_logs)
need(not experiment_log_frame.empty, 'Search calibration produced no experiment logs.')
experiment_log_frame.to_parquet(EXPERIMENT_LOG_PATH, index=False)

stability_rows = []
for (config_id_value, fold_index), group in experiment_log_frame.groupby(['config_id', 'fold_index'], sort=True):
    repeat_groups = []
    runtime_totals = []
    verifier_means = []
    for repeat_index, repeat_group in group.groupby('repeat_index', sort=True):
        repeat_groups.append(outcome_signature(repeat_group))
        runtime_totals.append(float(repeat_group['runtime_sec'].sum()))
        verifier_means.append(stable_mean(repeat_group['verifier_score']))
    stability_rows.append(
        {
            'config_id': config_id_value,
            'fold_index': int(fold_index),
            'stable_outcomes': len(set(repeat_groups)) == 1,
            'runtime_cv': runtime_cv(runtime_totals),
            'verifier_delta': max(verifier_means) - min(verifier_means) if verifier_means else 0.0,
        }
    )

stability_frame = pd.DataFrame(stability_rows)
need(not stability_frame.empty, 'Stability checks produced no rows.')
need(bool(stability_frame['stable_outcomes'].all()), 'Search calibration results are not stable across deterministic repeats.')
max_runtime_cv_observed = float(stability_frame['runtime_cv'].max())
need(max_runtime_cv_observed <= MAX_RUNTIME_CV, f'Runtime variation exceeded the allowed coefficient of variation: {max_runtime_cv_observed:.3f}')

repeat_agg = experiment_log_frame.groupby(['config_id', 'search_strategy', 'branch_width', 'depth', 'repair_attempts', 'retrieval_depth', 'fold_index', 'repeat_index'], sort=True).agg(
    problem_count=('problem_id', 'count'),
    solve_rate=('solved', 'mean'),
    runtime_mean_sec=('runtime_sec', 'mean'),
    runtime_total_sec=('runtime_sec', 'sum'),
    verifier_score_mean=('verifier_score', 'mean'),
    confidence_mean=('confidence', 'mean'),
    execution_failure_rate=('status', lambda values: sum(1 for value in values if value != 'ok') / max(1, len(values))),
    retrieval_hits_used_mean=('retrieval_hits_used', 'mean'),
).reset_index()

fold_agg = repeat_agg.groupby(['config_id', 'search_strategy', 'branch_width', 'depth', 'repair_attempts', 'retrieval_depth', 'fold_index'], sort=True).agg(
    solve_rate=('solve_rate', 'mean'),
    runtime_mean_sec=('runtime_mean_sec', 'mean'),
    verifier_score_mean=('verifier_score_mean', 'mean'),
    confidence_mean=('confidence_mean', 'mean'),
    execution_failure_rate=('execution_failure_rate', 'mean'),
).reset_index()

summary_frame = fold_agg.groupby(['config_id', 'search_strategy', 'branch_width', 'depth', 'repair_attempts', 'retrieval_depth'], sort=True).agg(
    fold_count=('fold_index', 'nunique'),
    solve_rate_mean=('solve_rate', 'mean'),
    solve_rate_min=('solve_rate', 'min'),
    solve_rate_max=('solve_rate', 'max'),
    runtime_mean_sec=('runtime_mean_sec', 'mean'),
    verifier_score_mean=('verifier_score_mean', 'mean'),
    confidence_mean=('confidence_mean', 'mean'),
    execution_failure_rate_mean=('execution_failure_rate', 'mean'),
).reset_index()
summary_frame['solve_rate_fold_gap'] = summary_frame['solve_rate_max'] - summary_frame['solve_rate_min']
summary_frame['generalizes_across_folds'] = summary_frame['solve_rate_fold_gap'] <= MAX_FOLD_SOLVE_GAP
summary_frame['stable_across_repeats'] = summary_frame['config_id'].isin(stability_frame[stability_frame['stable_outcomes']]['config_id'])
summary_frame['pareto_optimal'] = [is_pareto_optimal(summary_frame, index) for index in summary_frame.index]
summary_frame['tradeoff_score'] = tradeoff_score(summary_frame)

stable_generalizing = summary_frame[(summary_frame['stable_across_repeats']) & (summary_frame['generalizes_across_folds'])].copy()
need(not stable_generalizing.empty, 'No tuned config remained stable and cross-fold generalizing; calibration would overfit the subset.')

recommended_pool = stable_generalizing[stable_generalizing['pareto_optimal']].copy()
if recommended_pool.empty:
    recommended_pool = stable_generalizing.copy()
recommended_pool = recommended_pool.sort_values(['tradeoff_score', 'solve_rate_mean', 'verifier_score_mean', 'runtime_mean_sec'], ascending=[False, False, False, True], kind='stable').reset_index(drop=True)
recommended_row = recommended_pool.iloc[0]

axis_summaries = {}
for parameter_name in ('branch_width', 'depth', 'repair_attempts', 'retrieval_depth'):
    axis_frame = stable_generalizing.groupby(parameter_name, sort=True).agg(
        solve_rate_mean=('solve_rate_mean', 'mean'),
        runtime_mean_sec=('runtime_mean_sec', 'mean'),
        verifier_score_mean=('verifier_score_mean', 'mean'),
        config_count=('config_id', 'nunique'),
    ).reset_index()
    axis_summaries[parameter_name] = diminishing_returns(axis_frame, parameter_name)

need(any(item['diminishing_returns_from'] is not None for item in axis_summaries.values()), 'Diminishing returns were not identified in the search sweep.')

summary_frame.to_parquet(CALIBRATION_RESULTS_PATH, index=False)

recommended_overrides = {
    'run_id': RUN_ID,
    'recommended_config_id': str(recommended_row['config_id']),
    'selection_policy': 'stable_generalizing_pareto_tradeoff',
    'route_overrides': {
        'branch_budget': int(recommended_row['branch_width']),
        'retrieval_depth': int(recommended_row['retrieval_depth']),
        'budget_plan': {
            'branch_budget': int(recommended_row['branch_width']),
            'retrieval_depth': int(recommended_row['retrieval_depth']),
            'max_search_depth': int(recommended_row['depth']),
            'repair_budget': int(recommended_row['repair_attempts']),
            'self_consistency_samples': int(recommended_row['branch_width']),
            'max_search_nodes': int(max(int(recommended_row['branch_width']) * max(2, int(recommended_row['depth'])) * 2, int(recommended_row['branch_width']) * 4)),
        },
    },
    'controller_overrides': {
        'deterministic': True,
        'self_consistency_samples': int(recommended_row['branch_width']),
        'frontier_width': int(recommended_row['branch_width']),
        'max_search_depth': int(recommended_row['depth']),
        'repair_budget': int(recommended_row['repair_attempts']),
        'max_search_nodes': int(max(int(recommended_row['branch_width']) * max(2, int(recommended_row['depth'])) * 2, int(recommended_row['branch_width']) * 4)),
    },
    'expected_tradeoff': {
        'solve_rate_mean': float(recommended_row['solve_rate_mean']),
        'verifier_score_mean': float(recommended_row['verifier_score_mean']),
        'runtime_mean_sec': float(recommended_row['runtime_mean_sec']),
        'solve_rate_fold_gap': float(recommended_row['solve_rate_fold_gap']),
    },
}
OVERRIDES_PATH.write_text(json.dumps(recommended_overrides, indent=2, ensure_ascii=True, sort_keys=True), encoding='utf-8')

summary_payload = {
    'run_id': RUN_ID,
    'dataset_path': str(DATASET_PATH),
    'dataset_schema': list(schema),
    'sample_size_requested': SAMPLE_SIZE,
    'sample_size_executed': len(sampled_rows),
    'num_folds': NUM_FOLDS,
    'repeats': REPEATS,
    'experiment_count': len(experiment_configs),
    'grid_experiment_count': sum(1 for item in experiment_configs if item['search_strategy'] == 'grid'),
    'random_experiment_count': sum(1 for item in experiment_configs if item['search_strategy'] == 'random'),
    'retrieval_index_path': str(INDEX_PATH),
    'stability_checks': stability_frame.to_dict(orient='records'),
    'axis_summaries': axis_summaries,
    'recommended_overrides_path': str(OVERRIDES_PATH),
    'recommended_config_id': str(recommended_row['config_id']),
    'recommended_expected_tradeoff': recommended_overrides['expected_tradeoff'],
    'pareto_optimal_configs': summary_frame[summary_frame['pareto_optimal']].sort_values(['tradeoff_score', 'solve_rate_mean', 'runtime_mean_sec'], ascending=[False, False, True], kind='stable')['config_id'].tolist(),
}
SUMMARY_PATH.write_text(json.dumps(summary_payload, indent=2, ensure_ascii=True, sort_keys=True), encoding='utf-8')
summary_payload


In [ ]:
summary_payload = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
recommended_overrides = json.loads(OVERRIDES_PATH.read_text(encoding='utf-8'))
calibration_results = pd.read_parquet(CALIBRATION_RESULTS_PATH)

final_report = {
    'recommended_config_id': summary_payload['recommended_config_id'],
    'recommended_expected_tradeoff': summary_payload['recommended_expected_tradeoff'],
    'diminishing_returns': {
        key: value['diminishing_returns_from']
        for key, value in summary_payload['axis_summaries'].items()
    },
    'recommended_route_overrides': recommended_overrides['route_overrides'],
    'recommended_controller_overrides': recommended_overrides['controller_overrides'],
}

print(json.dumps(final_report, indent=2, ensure_ascii=True, sort_keys=True))
calibration_results.sort_values(['tradeoff_score', 'solve_rate_mean', 'runtime_mean_sec'], ascending=[False, False, True], kind='stable').head(20)
